<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 12 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Publish a View to a Role, and Audit Who Read What</div>
</div>

Module 11 published `bi_order_metrics_l2`, and the BI team asks for an account. You grant a role read access to that view and nothing else, prove as a temporary ordinary user that the view is readable and its sources are not, and retire the access again. You find the account that Module 11's optional dashboard was granted directly, and move it to a role. Then you release a table under a stable name with a validated cutover and rollback, limit two regional roles to their own orders with Row Policies, keep the payment columns from a third role with column privileges, and find the ordinary users' queries, a refusal among them, in Doris's audit table. Plan about 75 minutes.

**Module 12: Publishing, Permissions, and Audit** · [Lesson](course.md) · [Quiz](quiz12_publishing_permissions_audit.ipynb) · [Course README](../../README.md)

## Before you begin

Complete Labs 9 and 11 first, in the same database. This Lab publishes Lab 11's `bi_order_metrics_l2` and tests access to its sources, Lab 10's `daily_order_metrics_l2` and Lab 9's `orders_fact_l2`; it changes none of them. If one is missing, the setup cell stops and says which Lab to finish.

Roles, users, and Row Policies belong to the whole Doris cluster, not to your lab database. The Lab creates roles with fixed names, `course_bi_reader_l3`, `course_region_east_l3`, `course_region_west_l3`, `course_orders_reader_l3`, and `course_shared_reader_l3`, and temporary ordinary users with random names, whose passwords stay in memory and are never shown; it removes them again, also when a cell fails. It keeps two things: the tables `published_orders_l3` and `candidate_orders_l3` of section 6, and, if Lab 11's optional dashboard account exists, the role `course_bi_dashboard_l3` that section 5 moves it to, so that the dashboard keeps working.

The users have the host `'%'`, which suits a personal sandbox; a production account needs its own host and credential design. Root runs the drills as the administrator; it is not the identity that tests what a consumer can read.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

import secrets

import pymysql

from dw_course.access import temporary_user
from dw_course.docker_runtime import connect_sandbox
from dw_course.operations import audit_records
from dw_course.runtime import expect
from dw_course.ui import card, show_records, show_sql

lab = connect_sandbox(module=12)


def roles():
    """SHOW ROLES, one dict per role, by name."""
    with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
        cursor.execute("SHOW ROLES")
        return {row["Name"]: row for row in cursor.fetchall()}


def grants(statement):
    """SHOW GRANTS or SHOW ALL GRANTS, one dict per account."""
    with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
        cursor.execute(statement)
        return cursor.fetchall()

## 1. Check the identity, the release object, and the role baseline

Before a change, record who makes it, on which object, and how things are now. The cell first removes the Lab's role from an earlier run, so its baseline is the state after that reset; a production change records the earlier state before it removes anything. Then it shows the current user and database and the grantor's own privileges, checks that the course role is absent, and checks the object to publish: the semantic view's 2 days, 10 orders, and 1310.00.

In [ ]:
lab.execute("DROP ROLE IF EXISTS course_bi_reader_l3")
lab.sql("SELECT CURRENT_USER() AS current_user, DATABASE() AS current_database", title="Who makes the change, and where")
lab.sql("SHOW GRANTS", title="The grantor's own privileges", columns=["UserIdentity", "Roles", "GlobalPrivs", "DatabasePrivs"])
expect("course_bi_reader_l3" in roles(), False, title="The course role does not exist yet")
expect(lab.query("SELECT COUNT(*), SUM(order_count), SUM(gross_amount) FROM bi_order_metrics_l2"), [(2, 10, "1310.00")],
       title="The release object: 2 days, 10 orders, 1310.00")

**Expected result**

`'root'@'%'` in your lab database; root's privileges: the role `operator` with `Node_priv,Admin_priv`, and read access to `information_schema` and `mysql`. Two green checks: the course role does not exist yet, and the view has 2 days, 10 orders, and 1310.00.

Root can create roles and grant anything, which is why it runs the change. A dashboard needs none of that: what it needs is decided next, on the role.

## 2. Grant a role read access to the semantic view only

Doris grants privileges to roles and roles to users: five people of the BI team then share one role, and when one leaves, only a membership changes. The role gets `SELECT_PRIV`, the privilege to read, on one object: `internal.<your database>.bi_order_metrics_l2`. Not the database, whose new tables would become readable too, and not the view's sources, which would let a consumer bypass the view.

`SHOW ROLES` then shows the role's privileges by scope. The second query is an example of a change record, with an ID, the role, the object, the privilege, and the reason: it documents the change; it is neither the approval nor an audit record that the `GRANT` ran.

In [ ]:
grant_sql = f"GRANT SELECT_PRIV ON internal.{lab.database}.bi_order_metrics_l2 TO ROLE 'course_bi_reader_l3'"
show_sql("The role and its grant", "CREATE ROLE course_bi_reader_l3;\n" + grant_sql + ";")
lab.execute("CREATE ROLE course_bi_reader_l3")
lab.execute(grant_sql)
show_records("The role after the grant", [{key: roles()["course_bi_reader_l3"][key] for key in
                                           ("Name", "Users", "GlobalPrivs", "DatabasePrivs", "TablePrivs")}])
role = roles()["course_bi_reader_l3"]
expect((role["GlobalPrivs"], role["DatabasePrivs"], role["TablePrivs"]),
       (None, None, f"internal.{lab.database}.bi_order_metrics_l2: Select_priv"),
       title="The role can read the view, and nothing else")
lab.sql("""SELECT 'lab12-example' AS change_id, 'course_bi_reader_l3' AS role_name,
       'bi_order_metrics_l2' AS object_name, 'SELECT_PRIV' AS privilege,
       'course exercise: read-only dashboard role' AS change_reason""", title="An example change record, not an audit record");

**Expected result**

The two statements, then the role: no users yet, no global or database privileges, and `TablePrivs` `internal.<your database>.bi_order_metrics_l2: Select_priv`. A green check, then the example change record in one row.

A privilege in `GlobalPrivs`, `CatalogPrivs`, or `DatabasePrivs` would cover more than the one view. `Select_priv` lets the role read; it does not let it grant (`Grant_priv`) or administer (`Admin_priv`).

## 3. Test the role as an ordinary user

`SHOW ROLES` shows the configuration; only a query as the consumer shows what the consumer can do. `temporary_user()` creates an ordinary user with a random name and password, grants it the role, and drops it at the end of the block, also when a check fails. As that user, the cell reads the view, tries the view's two sources, and, after the role is revoked from the user, tries the view again. Predict which of the four queries Doris refuses.

In [ ]:
answers = {}
with temporary_user(lab, "course_bi_reader_l3") as reader:
    expect(reader.query(f"SELECT CAST(order_date AS STRING), order_count, gross_amount FROM {lab.database}.bi_order_metrics_l2 ORDER BY order_date"),
           [("2026-01-01", 9, "1200.00"), ("2026-01-02", 1, "110.00")], title="The user reads the view")
    for source in ("daily_order_metrics_l2", "orders_fact_l2"):
        answers[f"{source} (a source of the view)"] = reader.denied(f"SELECT COUNT(*) FROM {lab.database}.{source}")
    expect(all(answers.values()), True, title="The user cannot read the view's sources")
    lab.execute(f"REVOKE 'course_bi_reader_l3' FROM {reader.identity}")
    answers["bi_order_metrics_l2, after REVOKE"] = reader.denied(f"SELECT COUNT(*) FROM {lab.database}.bi_order_metrics_l2")
    expect(answers["bi_order_metrics_l2, after REVOKE"] is not None, True, title="Without the role, the view is refused too")
show_records("What Doris answered the user", [{"query": query, "answer": answer} for query, answer in answers.items()])

**Expected result**

The statements that create the temporary user, then three green checks: the user reads the view's two rows, cannot read its two sources, and after the `REVOKE` cannot read the view either. Doris's answers: for each source, `Access denied; you need (at least one of) the (Admin_priv,Select_priv) privilege(s) on table daily_order_metrics_l2 for this operation` (or `orders_fact_l2`); after the `REVOKE`, `Access denied for user 'course_reader_l3_<random>' to database '<your database>'`.

The view is readable although its sources are not: Doris checks the privilege on the view that the user names. Without the role, the user has no privilege left in your lab database, so Doris refuses even the connection to it. The user is dropped at the end of the cell.

## 4. Retire the access

The dashboard is retired, and the role has no members left: the temporary user is gone. Retiring the role takes two statements, in this order: revoke its privilege, then drop it. The view and its rows are not touched.

In [ ]:
retire_sql = [f"REVOKE SELECT_PRIV ON internal.{lab.database}.bi_order_metrics_l2 FROM ROLE 'course_bi_reader_l3'",
              "DROP ROLE IF EXISTS course_bi_reader_l3"]
show_sql("Retire the role", ";\n".join(retire_sql) + ";")
for statement in retire_sql:
    lab.execute(statement)
expect("course_bi_reader_l3" in roles(), False, title="The role is gone")
expect(lab.query("SELECT COUNT(*), SUM(order_count), SUM(gross_amount) FROM bi_order_metrics_l2"), [(2, 10, "1310.00")],
       title="The view is unchanged")

**Expected result**

The two statements and two green checks: the role is gone, and the view still has 2 days, 10 orders, and 1310.00.

Had another dashboard still used the role, revoking its privilege would have cut that dashboard off too; the practice retires one of two dashboards that share a role. Keep the approval and the change record of the retirement: `DROP ROLE` removes an object, it does not record why.

## 5. Move a direct grant to a role

Module 11's optional dashboard created a Doris account and granted it the payment view directly, to the user. A direct grant works, but it does not appear in the roles that a review reads, and the next account for the same dashboard needs the same grant again.

`SHOW ALL GRANTS` lists every account with its privileges, its own merged with its roles': the output does not say which is which. An account without roles has only its own. The cell lists the accounts without roles that hold privileges on the tables of your lab database. If it finds the dashboard's account, `course_bi_<suffix>`, it moves the grant to a role, `course_bi_dashboard_l3`: grant the role the privilege, grant the account the role, then revoke the account's own privilege. In that order the account never loses access, and the dashboard keeps working.

Doris keeps an account's own privileges in a hidden role of the account, `default_role_rbac_<user>@<host>`. Revoking a privilege that is not there fails and names that role, so a second `REVOKE` proves that no direct grant is left. If you did not build the dashboard, there is nothing to move, and the cell says so.

In [ ]:
database_grants = [row for row in grants("SHOW ALL GRANTS")
                   if f"internal.{lab.database}." in (row["TablePrivs"] or "")]
direct = [row for row in database_grants if not row["Roles"]]
dashboard_accounts = [row for row in database_grants if row["UserIdentity"].startswith("'course_bi_")]
show_records("Accounts without roles that hold privileges in this database",
             [{"account": row["UserIdentity"], "table_privileges": row["TablePrivs"]} for row in direct],
             columns=["account", "table_privileges"])
dashboard = [row["UserIdentity"] for row in direct if row["UserIdentity"].startswith("'course_bi_")]
if dashboard:
    account = dashboard[0]
    revoke_direct = f"REVOKE SELECT_PRIV ON internal.{lab.database}.bi_paid_kpis_l2 FROM {account}"
    move_sql = [
        "CREATE ROLE IF NOT EXISTS course_bi_dashboard_l3",
        f"GRANT SELECT_PRIV ON internal.{lab.database}.bi_paid_kpis_l2 TO ROLE 'course_bi_dashboard_l3'",
        f"GRANT 'course_bi_dashboard_l3' TO {account}",
        revoke_direct,
    ]
    show_sql("Move the dashboard account's grant to a role", ";\n".join(move_sql) + ";")
    for statement in move_sql:
        lab.execute(statement)
    after = grants(f"SHOW GRANTS FOR {account}")[0]
    show_records("The dashboard account after the move", [{key: after[key] for key in ("UserIdentity", "Roles", "TablePrivs")}])
    try:
        lab.execute(revoke_direct)
        left = "A direct grant was still there; this REVOKE removed it"
    except pymysql.err.OperationalError as error:
        left = None if "does not exist in role default_role_rbac" in str(error) else str(error)
    expect((after["Roles"], left), ("course_bi_dashboard_l3", None),
           title="The account reads through the role, and no direct grant is left")
elif any("course_bi_dashboard_l3" in (row["Roles"] or "").split(",") for row in dashboard_accounts):
    show_records("The dashboard already reads through its role",
                 [{key: row[key] for key in ("UserIdentity", "Roles", "TablePrivs")} for row in dashboard_accounts])
    card("The dashboard account already reads through course_bi_dashboard_l3. No grants were changed. "
         "Refresh the saved Superset charts to check access.", "ok", "Already migrated")
elif dashboard_accounts:
    show_records("Inspect the dashboard account's existing roles before changing grants", dashboard_accounts)
    card("A dashboard account with roles already exists in this database. No grants were changed; "
         "inspect its roles and TablePrivs above before changing access.", "skip", "Review existing roles")
else:
    card("No course dashboard account has table privileges in this database. Module 11's optional "
         "dashboard was not built here, so there is nothing to move.", "skip", "No dashboard account")

**Expected result**

If you built Module 11's optional dashboard: one account, `'course_bi_<suffix>'@'%'`, with `internal.<your database>.bi_paid_kpis_l2: Select_priv`; the four statements; then the account with the role `course_bi_dashboard_l3` and still `bi_paid_kpis_l2: Select_priv` in `TablePrivs`, now from the role; and a green check: the second `REVOKE` failed with `... does not exist in role default_role_rbac_course_bi_<suffix>@%`, so no direct grant is left. Refresh the dashboard in Superset: its charts still load, now through the role. Run again: the direct-grant table is empty, the account still appears with `course_bi_dashboard_l3`, and an **Already migrated** card says no grants were changed. The existing dashboard still works; do not rebuild it.

If you have not built the optional dashboard in this database: an empty table and a gray **No dashboard account** card. An account with other existing roles is shown separately for inspection; this cell does not change those roles.

The account kept its access throughout: it had the role before its own grant was revoked. Revoked first, the dashboard would have failed in between.

## 6. Validate, cut over, and roll back a table release

A table-name replacement is atomic; a correct publication is a larger procedure. This controlled drill creates an old and a candidate table with the same contract, backfills the candidate, simulates a **paused writer**, applies its final delta to both, and compares every business row before release.

Using `swap=true` retains the old data under the candidate name. First rehearse rollback while writers remain paused. Then release again and resume one simulated write. **Predict:** is the retained old table now a safe rollback target? No: it lacks that new write. No real CDC pipeline is paused here; this demonstrates the cutoff requirement, not zero-downtime migration.

In [ ]:
for table in ("published_orders_l3", "candidate_orders_l3"):
    lab.execute(f"DROP TABLE IF EXISTS {table}")
    lab.execute(f"""CREATE TABLE {table} (
        order_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
    ) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
    PROPERTIES ("replication_num"="1")""")
lab.insert("published_orders_l3", ["order_id","amount"], [(7101,70),(7102,90)])
lab.execute("INSERT INTO candidate_orders_l3 SELECT * FROM published_orders_l3")
# Simulated final delta at a paused-writer boundary, applied before validation.
for table in ("published_orders_l3", "candidate_orders_l3"):
    lab.insert(table, ["order_id","amount"], [(7102,95),(7103,40)])
release_rows = [(7101,"70.00"),(7102,"95.00"),(7103,"40.00")]
expect(lab.query("SELECT * FROM candidate_orders_l3 ORDER BY order_id"), release_rows, title="Candidate reconciles against the final-delta ledger")
expect(lab.query("SELECT * FROM published_orders_l3 ORDER BY order_id"), release_rows, title="The published table holds the same final delta")
lab.execute('ALTER TABLE published_orders_l3 REPLACE WITH TABLE candidate_orders_l3 PROPERTIES("swap"="true")')
expect(lab.query("SELECT * FROM published_orders_l3 ORDER BY order_id"), release_rows, title="Stable consumer name after release")
# Rollback rehearsal BEFORE writers resume: both sides still contain the validated cutoff.
lab.execute('ALTER TABLE published_orders_l3 REPLACE WITH TABLE candidate_orders_l3 PROPERTIES("swap"="true")')
expect(lab.query("SELECT * FROM published_orders_l3 ORDER BY order_id"), release_rows, title="Paused-writer rollback reconciles")
lab.execute('ALTER TABLE published_orders_l3 REPLACE WITH TABLE candidate_orders_l3 PROPERTIES("swap"="true")')
lab.insert("published_orders_l3", ["order_id","amount"], [(7104,25)])
expect(lab.query("SELECT COUNT(*),SUM(amount) FROM published_orders_l3"), [(4,"230.00")])
expect(lab.query("SELECT COUNT(*),SUM(amount) FROM candidate_orders_l3"), [(3,"205.00")], title="Retained old data lacks the post-release write; do not swap it back blindly")
lab.sql("SELECT 'current publication' AS data_copy, COUNT(*) AS orders,SUM(amount) AS amount FROM published_orders_l3 UNION ALL SELECT 'retained cutoff',COUNT(*),SUM(amount) FROM candidate_orders_l3", title="Rollback safety depends on the write boundary");

**Expected result**

Five green checks: the candidate matches the final ledger of 7101 (70.00), 7102 (95.00), and 7103 (40.00); so does the published table; the name still answers after the release; the rollback rehearsal, with writers still paused, gives the same rows; and after the release and one more write, the retained table lacks that write. The last table: the current publication has 4 orders and 230.00, the retained copy 3 and 205.00.

The swap is atomic, so readers never see half a table. The procedure around it makes it safe: a paused writer, a reconciled final delta, and a rollback that is only safe while nothing has been written since the release.

## 7. Limit each region to its own orders with Row Policies

The EAST team may read only EAST orders, the WEST team only WEST ones. A table privilege cannot say that: it covers every row. A **Row Policy** adds a condition that Doris applies to every query of the table by the users or roles it names; a `RESTRICTIVE` policy keeps only the rows that match. Each region gets a role with `SELECT_PRIV` on `orders_fact_l2` and a policy on that role, so the policy follows the role's members, as the privilege does.

As each region's temporary user, the cell reads the orders per region, asks explicitly for the other region's orders, and runs one query with a tag, which section 9 finds in the audit table. Testing as root would prove nothing: the policies do not apply to root.

In [ ]:
run_tag = "course_audit_" + secrets.token_hex(6)
audited_users = []
regions = {"EAST": "course_region_east_l3", "WEST": "course_region_west_l3"}
policy_sql = []
for region, role in regions.items():
    policy_sql += [f"CREATE ROLE {role}",
                   f"GRANT SELECT_PRIV ON internal.{lab.database}.orders_fact_l2 TO ROLE '{role}'",
                   f"CREATE ROW POLICY region_{region.lower()}_l3 ON {lab.database}.orders_fact_l2 "
                   f"AS RESTRICTIVE TO ROLE {role} USING (region = '{region}')"]
show_sql("Two roles, each with a Row Policy", ";\n".join(policy_sql) + ";")


def drop_region_objects():
    # A dropped role leaves its Row Policies behind, so drop the policies first.
    for region, role in regions.items():
        lab.execute(f"DROP ROW POLICY IF EXISTS region_{region.lower()}_l3 ON {lab.database}.orders_fact_l2")
        lab.execute(f"DROP ROLE IF EXISTS {role}")


drop_region_objects()
visible = []
try:
    for statement in policy_sql:
        lab.execute(statement)
    for region, role in regions.items():
        with temporary_user(lab, role, prefix="course_region_l3") as reader:
            rows = reader.query("SELECT region, COUNT(*), SUM(order_amount) FROM orders_fact_l2 GROUP BY region")
            other = "WEST" if region == "EAST" else "EAST"
            expect((rows, reader.query("SELECT COUNT(*) FROM orders_fact_l2 WHERE region = %s", (other,))),
                   ([(region, 6, "830.00")] if region == "EAST" else [(region, 5, "680.00")], [(0,)]),
                   title=f"The {region} user sees only {region} orders, even when it asks for {other}")
            reader.query(f"SELECT '{run_tag}_{region}' AS request_tag, COUNT(*) AS orders FROM orders_fact_l2")
            audited_users.append(reader.name)
            visible.append({"user": reader.name, "role": role, "region": rows[0][0], "orders": rows[0][1], "amount": str(rows[0][2])})
finally:
    drop_region_objects()
show_records("What each regional user could see", visible)
expect([row[0] for row in lab.query("SHOW ROW POLICY") if row[2] == lab.database and row[3] == "orders_fact_l2"], [],
       title="No Row Policy is left on orders_fact_l2")

**Expected result**

The statements, then two green checks: the EAST user sees 6 orders and 830.00 and nothing of WEST, the WEST user 5 orders and 680.00 and nothing of EAST. The table of what each user could see, and a last green check: no policy on `orders_fact_l2` is left.

The `WHERE region = 'WEST'` of the EAST user returned 0 orders: the policy is applied in the database, so no query can widen it, unlike a filter in a dashboard. The cell drops the policies before the roles, because dropping a role does not drop its policies: a role created later with the same name would get them back.

## 8. Keep the payment columns from a role with column privileges

Only Finance may see what was paid and refunded. A role for everyone else gets `SELECT_PRIV` on five columns of `orders_fact_l2`: `order_id`, `order_date`, `status`, `order_amount`, and `region`. As its user, the cell reads those columns, then tries `paid_amount`, `SELECT *`, and `COUNT(*)`. Predict which ones Doris refuses. The first two queries carry the tag of section 9: one that runs and one that Doris refuses.

In [ ]:
column_sql = (f"GRANT SELECT_PRIV(order_id, order_date, status, order_amount, region) "
              f"ON internal.{lab.database}.orders_fact_l2 TO ROLE 'course_orders_reader_l3'")
show_sql("A role with five columns", "CREATE ROLE course_orders_reader_l3;\n" + column_sql + ";")
lab.execute("DROP ROLE IF EXISTS course_orders_reader_l3")
answers = {}
try:
    lab.execute("CREATE ROLE course_orders_reader_l3")
    lab.execute(column_sql)
    with temporary_user(lab, "course_orders_reader_l3", prefix="course_orders_l3") as reader:
        expect(reader.query(f"SELECT '{run_tag}_columns' AS request_tag, COUNT(order_id), SUM(order_amount) FROM orders_fact_l2"),
               [(f"{run_tag}_columns", 11, "1510.00")], title="The user reads the five columns")
        answers["SUM(paid_amount)"] = reader.denied(f"SELECT '{run_tag}_payments' AS request_tag, SUM(paid_amount) FROM orders_fact_l2")
        answers["SELECT *"] = reader.denied("SELECT * FROM orders_fact_l2")
        answers["COUNT(*)"] = reader.denied("SELECT COUNT(*) FROM orders_fact_l2")
        expect(all(answers.values()), True, title="The payment column, SELECT *, and COUNT(*) are refused")
        audited_users.append(reader.name)
finally:
    lab.execute("DROP ROLE IF EXISTS course_orders_reader_l3")
show_records("What Doris answered the user", [{"query": query, "answer": answer} for query, answer in answers.items()])

**Expected result**

Two green checks: the user reads 11 orders and 1510.00 from its columns, and Doris refuses the other three queries. Its answers: for `SUM(paid_amount)`, `Permission denied: user [...] does not have privilege for [priv predicate: OR, Admin_priv,Select_priv] command on [internal].[<your database>].[orders_fact_l2].[paid_amount]`; for `SELECT *`, the same for `[customer_id]`; for `COUNT(*)`, `Access denied; you need (at least one of) the (Admin_priv,Select_priv) privilege(s) on table orders_fact_l2 for this operation`.

`SELECT *` asks for every column, and Doris names the first one that the role lacks, `customer_id`. `COUNT(*)` reads no column, and Doris requires the privilege on the whole table for it; `COUNT(order_id)` counts the same rows with a granted column. A column grant covers the columns that exist now: a column added later is not in it, which suits a role that must not see new data by default, and needs a review when it should.

## 9. Find the users' queries in the audit table

Grants say what an identity may do; the results above show what each one saw; the audit says what actually ran. Doris's audit plugin writes each statement, with the user, the database, its state, and its error code, to the table `__internal_schema.audit_log`, which SQL can read. It writes in batches, at least once a minute, so the cell waits for the four tagged queries of the temporary users: the two regional queries, the query of the five columns, and the refused query of `paid_amount`.

The filter on the users keeps root's own statements out, and with them anything that root typed. The same records are also in the FE's log file `fe.audit.log`.

In [ ]:
audit_records(lab, audited_users, run_tag, expected=4);

**Expected result**

A progress card for up to about a minute, then four rows, one per tagged query: the two regional users' `SELECT '<tag>_EAST' ...` and `SELECT '<tag>_WEST' ...` and the five-column user's `SELECT '<tag>_columns' ...`, each with `state` EOF, `error_code` 0, and `return_rows` 1; and its `SELECT '<tag>_payments' ...` with `state` ERR, `error_code` 1105, and `return_rows` 0. Each row has the user, your lab database, and the query's ID.

EOF is the state of a query that returned its rows; ERR with an error code is a refusal or another error. The record says who ran what and what Doris answered. It does not prove that the policies are right, that every FE's records were collected, or who the person behind an account was: on a cluster, every FE writes its own records.

## 10. Practice: retire one of two dashboards that share a role

Dashboards A and B read `bi_order_metrics_l2` through the same role. A is retired; B stays. Create the role `course_shared_reader_l3` with `SELECT_PRIV` on the view and two temporary users, one per dashboard. Check that both read the view; retire A only; check that A is refused and B still reads. Remove the role at the end, also if a check fails.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
shared_grant = f"GRANT SELECT_PRIV ON internal.{lab.database}.bi_order_metrics_l2 TO ROLE 'course_shared_reader_l3'"
view = f"SELECT SUM(order_count) FROM {lab.database}.bi_order_metrics_l2"
lab.execute("DROP ROLE IF EXISTS course_shared_reader_l3")
try:
    lab.execute("CREATE ROLE course_shared_reader_l3")
    lab.execute(shared_grant)
    with temporary_user(lab, "course_shared_reader_l3", prefix="course_dashboard_a") as a, \
         temporary_user(lab, "course_shared_reader_l3", prefix="course_dashboard_b") as b:
        expect((a.query(view), b.query(view)), ([(10,)], [(10,)]), title="Both dashboards read the view")
        lab.execute(f"REVOKE 'course_shared_reader_l3' FROM {a.identity}")
        expect((a.denied(view) is not None, b.query(view)), (True, [(10,)]),
               title="A is refused; B still reads through the role")
finally:
    lab.execute("DROP ROLE IF EXISTS course_shared_reader_l3")

**Expected result**

The statements that create the two users, then two green checks: both read 10 orders; after the `REVOKE` of A's membership, A is refused and B still reads.

Revoking the role's privilege instead would have cut off B too. Before you retire access, look at the role's `Users` in `SHOW ROLES`, and for other paths: a direct grant or another role could still let A read.

## You are done when

- a role with `SELECT_PRIV` on `bi_order_metrics_l2` alone let an ordinary user read the view and not its sources, and you can say what `SHOW ROLES`, the user's queries, and the change record each show;
- you can say when to revoke a membership and when a role's privilege, and in which order to retire a role;
- you can move a direct grant to a role without the account losing access;
- you can say why the rollback of a released table is safe only before the next write;
- you can tell a Row Policy from a dashboard filter, and say why `COUNT(*)` failed with column privileges;
- you found the users' queries, the refusal too, in `__internal_schema.audit_log`, and can say what an audit record does not prove.

## Next

Open [Quiz 12: Publishing, Permissions, and Audit](quiz12_publishing_permissions_audit.ipynb), then continue to [Module 13: Storage and Lifecycle Management](../module13-storage-lifecycle/course.md).